# ¡Bienvenido a este Notebook Educativo sobre Predicción de Churn!

## ¿Qué aprenderás aquí?

Este notebook te guiará a través de un proyecto completo de **Machine Learning** aplicado a un problema real en el mundo de los negocios: predecir si un cliente de una empresa de telecomunicaciones abandonará el servicio (**churn**).

### Conceptos clave que cubriremos:
- **Análisis Exploratorio de Datos (EDA)**: Cómo explorar y entender tus datos antes de modelar.
- **Ingeniería de Características**: Preparar los datos para que los modelos aprendan mejor.
- **Modelos de Machine Learning**: Usar algoritmos avanzados como LightGBM, CatBoost y XGBoost.
- **Optimización de Hiperparámetros**: Encontrar los mejores ajustes para tus modelos usando Optuna.
- **Evaluación de Modelos**: Medir qué tan bien funciona tu modelo y por qué importa.
- **Técnicas Avanzadas**: Validación cruzada, stacking y manejo de datos desbalanceados.

### ¿Por qué es importante predecir el churn?
- Las empresas pierden millones al no retener clientes.
- Un buen modelo puede ayudar a identificar clientes en riesgo y ofrecerles incentivos.
- Aprenderás a aplicar ML en escenarios reales, no solo teoría.

¡Sigue las celdas en orden y experimenta ejecutándolas! Si algo no entiendes, revisa las explicaciones o investiga más. Recuerda: el aprendizaje es iterativo.

# Predicción de Churn en Telecomunicaciones

**Objetivo:** Construir un modelo de clasificación binaria que prediga si un cliente de una empresa de telecomunicaciones va a darse de baja (*churn*).

---

## Estructura del notebook

1. Librerías y configuración
2. Funciones auxiliares de EDA
3. Funciones de entrenamiento y evaluación
4. Carga de datos
5. Descripción del dataset
6. Análisis exploratorio (EDA)
7. Ingeniería de características
8. Entrenamiento y optimización de modelos
9. Evaluación y generación de predicciones


## 1. Librerías y configuración

Importamos todas las librerías necesarias para el análisis, visualización y modelado.
También definimos las constantes globales que se usarán a lo largo del notebook:
- `RANDOM_SEED`: semilla para reproducibilidad.
- `NUMERIC_CONTINUOUS`: columnas numéricas continuas del dataset.
- `CATEGORY_COLUMNS`: columnas categóricas.
- `TARGET`: nombre de la variable objetivo.


In [ ]:
#Para Google Colab, instalar las dependencias necesarias
!pip install catboost optuna
#Para entorno diferente a Google Colab, descomentar la siguiente línea e instalar las dependencias necesarias
#!pip install -r requirements.txt

In [ ]:
# Librerías de manipulación y análisis de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Tests estadísticos y métricas de distribución
from scipy.stats import skew, kurtosis, shapiro, normaltest, spearmanr, ttest_ind

# Utilidades del sistema
import warnings
import os
import re

# Herramientas de modelado de scikit-learn
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

# Modelos de gradient boosting
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# Optimización bayesiana de hiperparámetros
import optuna

# Utilidades de visualización en Jupyter
from IPython.display import display, Markdown

# Métricas de evaluación de clasificación
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import roc_curve, roc_auc_score

# Configuración de pandas para mostrar todas las filas y columnas
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Suprimir warnings no críticos durante la ejecución
warnings.filterwarnings("ignore")

In [ ]:
# Semilla global para reproducibilidad de todos los procesos aleatorios
RANDOM_SEED = 27

# Variables numéricas continuas del dataset
NUMERIC_CONTINUOUS = [
    'tenure',          # Meses que lleva el cliente con la compañía
    'MonthlyCharges',  # Cargo mensual actual
    'TotalCharges'     # Cargo total acumulado
]

# Variables categóricas del dataset
CATEGORY_COLUMNS = [
    'gender',            # Género del cliente
    'SeniorCitizen',     # Si es ciudadano senior (1/0)
    'Partner',           # Si tiene pareja
    'Dependents',        # Si tiene dependientes
    'PhoneService',      # Si tiene servicio telefónico
    'MultipleLines',     # Si tiene múltiples líneas
    'InternetService',   # Tipo de servicio de internet
    'OnlineSecurity',    # Si tiene seguridad online
    'OnlineBackup',      # Si tiene copia de seguridad online
    'DeviceProtection',  # Si tiene protección de dispositivo
    'TechSupport',       # Si tiene soporte técnico
    'StreamingTV',       # Si tiene streaming de TV
    'StreamingMovies',   # Si tiene streaming de películas
    'Contract',          # Tipo de contrato
    'PaperlessBilling',  # Si tiene facturación electrónica
    'PaymentMethod'      # Método de pago
]

# Nombre de la variable objetivo (1 = churn, 0 = no churn)
TARGET = 'churn_flag'

## ¿Qué es el Análisis Exploratorio de Datos (EDA)?

Antes de construir modelos, **debemos entender nuestros datos**. El EDA es como "conocer a tus datos" antes de una cita: miras sus características, buscas patrones y detectas problemas.

### ¿Por qué es crucial?
- **Descubre insights**: ¿Qué variables afectan el churn? ¿Hay datos faltantes o outliers?
- **Prepara el terreno**: Decide qué transformaciones aplicar (e.g., normalizar variables sesgadas).
- **Evita errores**: Un modelo malo puede venir de datos mal entendidos.

En esta sección, definimos funciones reutilizables para EDA. Piensa en ellas como herramientas en tu caja: las usarás una y otra vez en proyectos futuros.

### Tips educativos:
- Siempre grafica tus datos: "Una imagen vale más que mil estadísticas".
- Prueba hipótesis simples: ¿Los clientes con contratos largos churn menos?
- Documenta hallazgos: Anota qué ves y por qué importa.

## 2. Funciones auxiliares de EDA

Definimos las funciones que usaremos durante el análisis exploratorio.
Están agrupadas aquí para mantener el notebook limpio; se invocarán más adelante.


### `describe_dataset`
Muestra un resumen completo del DataFrame: dimensiones, uso de memoria, tipos de datos
y estadísticas descriptivas separadas por tipo de variable (numérica, categórica, datetime).


In [ ]:
def describe_dataset(df):
    """
    Resumen general de un DataFrame:
    dimensiones, memoria, tipos de datos y estadísticas descriptivas.
    """
    display(Markdown("## 📋 Dataset Overview"))
    print("--- Basic Dimensions & Memory ---")

    # Número de filas y columnas
    num_rows, num_cols = df.shape
    print(f"**Shape (Rows, Columns):** ({num_rows:,}, {num_cols:,})")

    # Uso de memoria total en MB
    mem_usage = df.memory_usage(deep=True).sum()
    mem_gbs = mem_usage / (1024**2)
    print(f"**Total Memory Usage:** {mem_gbs:.2f} MB")

    print("\n--- Feature Data Types and Counts ---")

    # Conteo de columnas por tipo de dato
    dtype_counts = df.dtypes.astype(str).value_counts().reset_index()
    dtype_counts.columns = ['Data_Type', 'Count']
    print(dtype_counts.to_markdown(index=False))

    display(Markdown("\n## 📊 Descriptive Statistics"))

    # Separar columnas por tipo para estadísticas diferenciadas
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    datetime_cols = df.select_dtypes(include=['datetime']).columns.tolist()

    # Estadísticas para variables numéricas (con IQR adicional)
    if numerical_cols:
        display(Markdown("### Numerical Features"))
        num_desc = df[numerical_cols].describe().T
        num_desc['IQR'] = num_desc['75%'] - num_desc['25%']  # Rango intercuartílico
        display(num_desc.style.format("{:,.2f}"))
        print(f"Found {len(numerical_cols)} numerical features.")

    # Estadísticas para variables categóricas
    if categorical_cols:
        display(Markdown("### Categorical / Object Features"))
        cat_desc = df[categorical_cols].describe().T
        display(cat_desc.style.format({"unique": "{:,}", "freq": "{:,}"}))
        print(f"Found {len(categorical_cols)} categorical/object features.")

    # Estadísticas para variables de fecha/hora
    if datetime_cols:
        display(Markdown("### Datetime Features"))
        dt_desc = df[datetime_cols].describe().T
        display(dt_desc)
        print(f"Found {len(datetime_cols)} datetime features.")


### `plot_categorical_distribution`
Dibuja la distribución porcentual de una variable categórica mediante un gráfico de barras.
Útil para detectar desequilibrios o categorías dominantes.


In [ ]:
def plot_categorical_distribution(df, column, sample=50000, top_n=15, figsize=(10, 5)):
    """
    Muestra la distribución porcentual de una variable categórica.

    Parámetros:
        df      : DataFrame de entrada
        column  : nombre de la columna categórica a visualizar
        sample  : número máximo de filas a muestrear (evita lentitud con datasets grandes)
        top_n   : número máximo de categorías a mostrar
        figsize : tamaño del gráfico
    """
    # Submuestreo para datasets grandes
    df = df.sample(min(sample, len(df)))

    # Eliminar valores nulos antes de calcular frecuencias
    data = df[column].dropna()

    # Calcular porcentajes normalizados y ordenar de mayor a menor
    perc = (data.value_counts(normalize=True) * 100).sort_values(ascending=False)

    # Limitar al top_n para evitar gráficos saturados
    if len(perc) > top_n:
        perc = perc.head(top_n)

    plt.figure(figsize=figsize)
    ax = sns.barplot(x=perc.index, y=perc.values, palette="viridis")

    # Anotar el porcentaje encima de cada barra
    for i, v in enumerate(perc.values):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Percentage (%)")
    plt.title(f"Percentage Distribution of '{column}'")
    plt.tight_layout()
    plt.show()


### `plot_binary_feature`
Visualiza la distribución de una variable binaria (0/1). Especialmente útil para
explorar la variable objetivo y detectar desbalanceo de clases.


In [ ]:
def plot_binary_feature(df, column, figsize=(6,4)):
    """
    Visualiza la distribución porcentual de una variable binaria (0/1 o True/False).
    Útil para detectar desbalanceo en la variable objetivo.
    """
    # Calcular porcentajes de cada clase
    counts = df[column].value_counts(normalize=True) * 100
    counts = counts.sort_index()  # Asegurar orden: 0 antes que 1

    plt.figure(figsize=figsize)
    sns.barplot(x=counts.index.astype(str), y=counts.values, palette=["#4C72B0", "#55A868"])

    # Anotar el porcentaje encima de cada barra
    for i, v in enumerate(counts.values):
        plt.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=11)

    plt.title(f"Binary Feature Distribution: {column}")
    plt.xlabel(f"{column} (0/1)")
    plt.ylabel("Percentage (%)")
    plt.ylim(0, 110)  # Margen superior para las etiquetas
    plt.tight_layout()
    plt.show()


### `numeric_vs_target`
Compara la distribución de una variable numérica entre las dos clases del target.
Incluye KDE, boxplot, diferencia de medias y tamaño del efecto (Cohen's d).


In [ ]:
def numeric_vs_target(df, feature, target, figsize=(10,5)):
    """
    Compara la distribución de una variable numérica entre clientes con y sin churn.
    Incluye: KDE, boxplot, diferencia de medias y tamaño del efecto (Cohen's d).
    """
    plt.figure(figsize=figsize)

    # ── Gráfico de densidad (KDE) por clase ──────────────────────────────
    # Permite ver si las distribuciones de ambas clases se solapan o se separan
    sns.kdeplot(data=df, x=feature, hue=target, fill=True, common_norm=False,
                palette=["#4C72B0", "#DD8452"], alpha=0.5)
    plt.title(f"Distribution of {feature} by {target}")
    plt.show()

    # ── Boxplot por clase ─────────────────────────────────────────────────
    # Muestra mediana, cuartiles y outliers para cada grupo
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x=target, y=feature, palette="Set2")
    plt.title(f"Boxplot of {feature} by {target}")
    plt.show()

    # ── Estadísticas comparativas ─────────────────────────────────────────
    group0 = df[df[target] == 0][feature].dropna()  # Clientes sin churn
    group1 = df[df[target] == 1][feature].dropna()  # Clientes con churn

    # Diferencia de medias entre grupos
    mean_diff = group1.mean() - group0.mean()

    # Desviación estándar pooled para el cálculo de Cohen's d
    pooled_std = np.sqrt((group0.var() + group1.var()) / 2)
    cohend = mean_diff / pooled_std  # Tamaño del efecto

    # Test de Welch (no asume igualdad de varianzas)
    t_stat, p_val = ttest_ind(group0, group1, equal_var=False)

    print(f"\n📌 **Stats for {feature} vs {target}:**")
    print(f"Mean (No Churn): {group0.mean():.2f}")
    print(f"Mean (Churn):    {group1.mean():.2f}")
    print(f"Mean Difference: {mean_diff:.2f}")
    print(f"Cohen's d:       {cohend:.3f}  → {'SMALL' if abs(cohend)<0.3 else 'MEDIUM' if abs(cohend)<0.8 else 'LARGE'} effect")
    print(f"T-test p-value:  {p_val:.3e}\n")


### `categorical_vs_target` y `binary_vs_target`
Calculan y visualizan la tasa de churn por categoría, permitiendo identificar
qué valores de una variable están más asociados con la baja del cliente.


In [ ]:
def categorical_vs_target(df, feature, target, sample=50000, normalize=True, figsize=(10,5)):
    """
    Visualiza la tasa de churn para cada categoría de una variable categórica.
    Permite identificar qué valores están más asociados con la baja del cliente.

    Parámetros:
        df        : DataFrame de entrada
        feature   : nombre de la variable categórica
        target    : nombre de la variable objetivo binaria
        sample    : número máximo de filas a muestrear
        normalize : no usado actualmente (reservado para extensiones)
        figsize   : tamaño del gráfico
    """
    # Submuestreo para datasets grandes
    df = df.sample(min(sample, len(df)))

    # Calcular la tasa media de churn por categoría, ordenada de mayor a menor
    proportions = (
        df.groupby(feature)[target]
          .mean()
          .sort_values(ascending=False)
          .reset_index()
    )
    # Convertir a string para evitar problemas con tipos numéricos en el eje X
    proportions[feature] = proportions[feature].astype(str)

    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=proportions,
        x=feature,
        y=target,
        palette="viridis"
    )

    # Anotar el porcentaje de churn encima de cada barra
    for i, v in enumerate(proportions[target]):
        ax.text(i, v + 0.01, f"{v*100:.1f}%", ha="center", fontsize=10)

    plt.title(f"Tasa de Churn por {feature}")
    plt.ylabel("Tasa de Churn")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    # Mostrar tabla numérica con los valores exactos
    print(proportions)


In [ ]:
def binary_vs_target(df, col, target):
    """
    Analiza la relación entre una variable binaria (0/1) y el target binario.
    Genera una tabla resumen y un gráfico de barras con tasas de churn anotadas.

    Parámetros:
        df     : DataFrame de entrada
        col    : nombre de la columna binaria (debe contener solo 0 y 1)
        target : nombre de la columna objetivo binaria
    """
    # ── Validación de entradas ────────────────────────────────────────────
    if df[col].nunique() > 2:
        raise ValueError(f"La columna {col} no es binaria.")
    if df[target].nunique() > 2:
        raise ValueError(f"El target {target} debe ser binario.")

    # ── Calcular tasa de churn por grupo (0 y 1) ─────────────────────────
    summary = (
        df.groupby(col)[target]
        .agg(
            count="count",       # Total de observaciones por grupo
            positives="sum",     # Número de churns por grupo
            mean_rate="mean"     # Tasa de churn por grupo
        )
        .reset_index()
    )
    # Convertir tasa a porcentaje para facilitar la visualización
    summary["mean_rate_pct"] = summary["mean_rate"] * 100

    # ── Métricas comparativas entre grupos ────────────────────────────────
    if set(summary[col]) == {0, 1}:
        p0 = summary.loc[summary[col] == 0, "mean_rate"].values[0]  # Tasa grupo 0
        p1 = summary.loc[summary[col] == 1, "mean_rate"].values[0]  # Tasa grupo 1
        summary["risk_ratio"] = p1 / p0 if p0 > 0 else None  # Riesgo relativo
        summary["delta_rate"] = p1 - p0                       # Diferencia absoluta
    else:
        summary["risk_ratio"] = None
        summary["delta_rate"] = None

    # ── Gráfico de barras con tasas anotadas ──────────────────────────────
    plt.figure(figsize=(8, 5))
    sns.barplot(data=summary, x=col, y="mean_rate_pct", palette="Blues_r")

    # Anotar el porcentaje encima de cada barra
    for idx, row in summary.iterrows():
        plt.text(
            idx, row["mean_rate_pct"] + 0.5,
            f"{row['mean_rate_pct']:.1f}%",
            ha="center", fontsize=12, fontweight="bold"
        )

    plt.title(f"{col} vs {target} (Comparación de Tasa de Churn)", fontsize=14)
    plt.xlabel(col)
    plt.ylabel(f"% {target}=1")
    plt.tight_layout()
    plt.show()

    return summary


### `analyze_numeric_distribution`
Analiza la distribución de variables numéricas: histograma, boxplot, asimetría (*skewness*),
curtosis y test de normalidad. Devuelve un DataFrame con el resumen estadístico.


In [ ]:
def analyze_numeric_distribution(df, variables=None, sample_size=300000):
    """
    Genera gráficos de distribución y diagnósticos estadísticos para variables numéricas.
    Incluye asimetría, curtosis, tests de normalidad e interpretación automática.

    Parámetros:
        df          : DataFrame de entrada
        variables   : lista de columnas a analizar (si None, detecta automáticamente)
        sample_size : tamaño máximo de muestra para los tests de normalidad

    Retorna:
        summary_df  : DataFrame con asimetría, curtosis, p-valor y observaciones
    """
    # Detectar automáticamente las columnas numéricas si no se especifican
    if variables is None:
        variables = df.select_dtypes(include=[np.number]).columns.tolist()

    insights = []

    for col in variables:
        # Convertir a numérico y eliminar nulos
        data = pd.to_numeric(df[col], errors="coerce").dropna()

        # Submuestreo para el test de normalidad (evita lentitud con datasets grandes)
        sample = data.sample(min(len(data), sample_size), random_state=42)

        # Calcular asimetría y curtosis
        col_skew = skew(data)
        col_kurt = kurtosis(data)

        # Test de normalidad: D'Agostino & Pearson (más robusto que Shapiro para n grande)
        try:
            stat, p_normal = normaltest(sample)
        except:
            # Fallback a Shapiro-Wilk si normaltest falla
            stat, p_normal = shapiro(sample)

        # ── Gráficos de distribución ──────────────────────────────────────
        plt.figure(figsize=(12, 5))

        # Histograma con curva de densidad
        plt.subplot(1, 2, 1)
        sns.histplot(data, kde=True, bins=30, color="steelblue")
        plt.title(f"Distribution of {col}")

        # Boxplot para detectar outliers
        plt.subplot(1, 2, 2)
        sns.boxplot(x=data, color="salmon")
        plt.title(f"Boxplot of {col}")

        plt.tight_layout()
        plt.show()

        # ── Interpretación automática ─────────────────────────────────────
        interpretation = []

        # Interpretación de la asimetría
        if col_skew > 1:
            interpretation.append("Asimetría positiva fuerte → cola derecha larga, posibles outliers. Considerar log-transform.")
        elif 0.5 < col_skew <= 1:
            interpretation.append("Asimetría positiva moderada → puede beneficiarse de una transformación suave.")
        elif -1 <= col_skew < -0.5:
            interpretation.append("Asimetría negativa moderada → los datos se concentran en valores altos.")
        elif col_skew < -1:
            interpretation.append("Asimetría negativa fuerte → cola izquierda larga, posible saturación inferior.")
        else:
            interpretation.append("Distribución aproximadamente simétrica.")

        # Interpretación de la curtosis
        if col_kurt > 3:
            interpretation.append("Curtosis alta → colas pesadas, muchos outliers (leptocúrtica).")
        elif col_kurt < -1:
            interpretation.append("Distribución muy plana → posible multimodalidad (platicúrtica).")
        else:
            interpretation.append("Curtosis razonable → sin comportamiento extremo en las colas.")

        # Resultado del test de normalidad
        if p_normal < 0.05:
            interpretation.append("Test de normalidad FALLIDO → la distribución no es normal.")
        else:
            interpretation.append("Test de normalidad SUPERADO → la distribución puede ser aproximadamente normal.")

        # Sugerencia de transformación según la asimetría
        if col_skew > 0.75:
            interpretation.append("Transformación sugerida: log(x + 1) o Box-Cox.")
        elif col_skew < -0.75:
            interpretation.append("Transformación sugerida: reflexión + log, o Winsorización.")

        insights.append({
            "variable": col,
            "skewness": col_skew,
            "kurtosis": col_kurt,
            "normality_p_value": p_normal,
            "insights": " | ".join(interpretation),
        })

    return pd.DataFrame(insights)


### `plot_correlation_matrix`
Genera un mapa de calor con la matriz de correlación de Pearson entre variables numéricas.
Ayuda a detectar multicolinealidad antes del modelado.


In [ ]:
def plot_correlation_matrix(df):
    """
    Genera un mapa de calor con la matriz de correlación de Pearson.
    Solo incluye variables numéricas. La mitad superior se oculta para evitar redundancia.
    """
    # Seleccionar solo columnas numéricas
    df = df[df.select_dtypes(include=[np.number]).columns.tolist()]

    # Calcular la matriz de correlación de Pearson
    corr_matrix = df.corr()

    # Máscara para ocultar el triángulo superior (evita duplicar información)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    plt.figure(figsize=(12, 8))

    # Mapa de calor con anotaciones numéricas
    sns.heatmap(corr_matrix, mask=mask, annot=True,
                cmap="coolwarm", fmt=".2f", linewidths=0.5)

    plt.title("Feature Correlation Heatmap")
    plt.show()


## 3. Funciones de entrenamiento y evaluación

Este bloque contiene toda la infraestructura de modelado:
- Validación cruzada estratificada.
- Funciones objetivo para Optuna (LightGBM, CatBoost, XGBoost).
- Construcción de modelos con los mejores hiperparámetros.
- Generación de predicciones OOF (*Out-Of-Fold*) y sobre test.
- Stacking con metamodelo de regresión logística.
- Visualización de importancia de características y métricas de evaluación.


### `evaluate_model_cv`
Evalúa un modelo mediante validación cruzada estratificada (`StratifiedKFold`).
Devuelve la media y desviación estándar del AUC-ROC entre los folds.
Gestiona automáticamente las columnas categóricas para CatBoost.


In [ ]:
def evaluate_model_cv(model, X, y, n_splits=3, random_state=RANDOM_SEED, metric=roc_auc_score):
    """
    Evalúa un modelo con validación cruzada estratificada.
    Retorna la media y desviación estándar del AUC-ROC entre los folds.

    Parámetros:
        model        : instancia del clasificador a evaluar
        X            : features de entrenamiento
        y            : variable objetivo
        n_splits     : número de folds de la validación cruzada
        random_state : semilla para reproducibilidad
        metric       : función de métrica a usar (por defecto roc_auc_score)
    """
    # Validación cruzada estratificada: mantiene la proporción de clases en cada fold
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = []

    # Detectar columnas categóricas una sola vez (necesario para CatBoost)
    cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        # CatBoost requiere indicar explícitamente las columnas categóricas
        if isinstance(model, CatBoostClassifier):
            if len(cat_features) > 0:
                model.fit(X_train, y_train, cat_features=cat_features)
            else:
                model.fit(X_train, y_train)
        else:
            model.fit(X_train, y_train)

        # Predecir probabilidades y calcular la métrica
        preds = model.predict_proba(X_valid)[:, 1]
        score = metric(y_valid, preds)
        scores.append(score)

    return np.mean(scores), np.std(scores)


## Introducción al Modelado de Machine Learning

¡Ahora viene la parte emocionante: construir modelos que predigan el futuro! Usaremos algoritmos de **Gradient Boosting** (LightGBM, CatBoost, XGBoost), que son potentes para problemas de clasificación como este.

### Conceptos clave:
- **Validación Cruzada**: Divide los datos en "folds" para probar el modelo en datos no vistos, evitando sobreajuste.
- **Optimización de Hiperparámetros**: Los modelos tienen "knobs" (e.g., profundidad del árbol). Optuna los ajusta automáticamente usando búsqueda inteligente.
- **Stacking**: Combina predicciones de varios modelos para un resultado mejor (como un equipo de expertos).
- **Métricas**: AUC-ROC mide qué tan bien separa el modelo clases. >0.8 es bueno; >0.9 es excelente.

### Lección educativa:
- No uses el dataset entero para entrenar: Siempre reserva un "test set" para evaluación final.
- El desbalanceo (más no-churn que churn) es común; técnicas como stratified sampling lo manejan.
- Experimenta: Cambia hiperparámetros y ve cómo afecta el rendimiento. ¡Aprende por qué!

Recuerda: Un modelo no es magia; es matemáticas aprendiendo patrones de tus datos.

### Funciones objetivo para Optuna
Cada función define el espacio de búsqueda de hiperparámetros para un modelo distinto.
Optuna las llama repetidamente para encontrar la combinación que maximiza el AUC.
- `objective_lgbm`: espacio de búsqueda para LightGBM.
- `objective_catboost`: espacio de búsqueda para CatBoost.
- `objective_xgb`: espacio de búsqueda para XGBoost.


In [ ]:
def objective_lgbm(trial, X, y, n_splits=3, random_state=RANDOM_SEED):
    """
    Función objetivo de Optuna para LightGBM.
    Define el espacio de búsqueda de hiperparámetros y retorna el AUC medio en CV.
    """
    params = {
        "objective": "binary",          # Clasificación binaria
        "metric": "auc",                 # Métrica de optimización
        "boosting_type": "gbdt",         # Gradient Boosting Decision Tree
        "n_estimators": trial.suggest_int("n_estimators", 140, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "num_leaves": trial.suggest_int("num_leaves", 200, 4000),       # Controla complejidad del árbol
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 250),  # Regularización
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),        # Fracción de filas por árbol
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),  # Fracción de columnas
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),   # Regularización L1
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True), # Regularización L2
        "random_state": random_state,
        "device": "cpu",
        "verbosity": -1,
        "n_jobs": -1
    }
    model = LGBMClassifier(**params)
    mean_score, _ = evaluate_model_cv(model, X, y, n_splits=n_splits, random_state=random_state)
    return mean_score


In [ ]:
def objective_catboost(trial, X, y, n_splits=3, random_state=RANDOM_SEED):
    """
    Función objetivo de Optuna para CatBoost.
    Define el espacio de búsqueda de hiperparámetros y retorna el AUC medio en CV.
    """
    params = {
        "loss_function": "Logloss",      # Función de pérdida para clasificación binaria
        "eval_metric": "AUC",            # Métrica de evaluación
        "iterations": trial.suggest_int("iterations", 150, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),                     # Profundidad del árbol
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),  # Regularización L2
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),  # Aleatoriedad en splits
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),  # Intensidad del bagging
        "border_count": trial.suggest_int("border_count", 32, 255),     # Número de bordes para variables numéricas
        "verbose": 0,
        "task_type": "CPU",
        "random_seed": random_state
    }
    model = CatBoostClassifier(**params)
    mean_score, _ = evaluate_model_cv(model, X, y, n_splits=n_splits, random_state=random_state)
    return mean_score


In [ ]:
def objective_xgb(trial, X, y, n_splits=3, random_state=RANDOM_SEED):
    """
    Función objetivo de Optuna para XGBoost.
    Define el espacio de búsqueda de hiperparámetros y retorna el AUC medio en CV.
    """
    params = {
        "objective": "binary:logistic",  # Clasificación binaria con salida probabilística
        "eval_metric": "auc",
        "n_estimators": trial.suggest_int("n_estimators", 200, 400),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20),  # Regularización mínima por hoja
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-4, 10.0, log=True),    # Ganancia mínima para hacer un split
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),   # Regularización L1
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True), # Regularización L2
        "random_state": random_state,
        "enable_categorical": False,  # Desactivado: las categóricas ya están codificadas
        "n_jobs": -1
    }
    model = XGBClassifier(**params)
    mean_score, _ = evaluate_model_cv(model, X, y, n_splits=n_splits, random_state=random_state)
    return mean_score


### `tune_model`
Lanza el proceso de optimización de Optuna para cualquier función objetivo.
Devuelve el estudio con los mejores parámetros encontrados.


In [ ]:
def tune_model(objective_fn, X, y, n_trials=3, study_name="study"):
    """
    Lanza la optimización bayesiana de Optuna para una función objetivo dada.

    Parámetros:
        objective_fn : función objetivo (objective_lgbm, objective_catboost, objective_xgb)
        X            : features de entrenamiento
        y            : variable objetivo
        n_trials     : número de combinaciones de hiperparámetros a probar
        study_name   : nombre identificativo del estudio

    Retorna:
        study : objeto Optuna con el historial de trials y los mejores parámetros
    """
    # Crear estudio de maximización (queremos maximizar el AUC)
    study = optuna.create_study(direction="maximize", study_name=study_name)

    # Ejecutar la optimización: Optuna llama a objective_fn n_trials veces
    study.optimize(lambda trial: objective_fn(trial, X, y), n_trials=n_trials)

    print(f"Mejor AUC para {study_name}: {study.best_value:.6f}")
    print("Mejores parámetros:")
    print(study.best_params)
    return study


### Constructores de modelos (`build_lgbm`, `build_catboost`, `build_xgb`)
Instancian cada modelo con los hiperparámetros óptimos obtenidos por Optuna,
añadiendo los parámetros fijos necesarios (función de pérdida, semilla, etc.).


In [ ]:
def build_lgbm(best_params, random_state=RANDOM_SEED):
    """
    Construye un LGBMClassifier con los mejores hiperparámetros de Optuna.
    Añade los parámetros fijos de configuración (objetivo, métrica, dispositivo).
    """
    params = best_params.copy()  # Evitar modificar el dict original
    params.update({
        "objective": "binary",    # Clasificación binaria
        "metric": "auc",          # Métrica de evaluación
        "boosting_type": "gbdt",  # Tipo de boosting
        "random_state": random_state,
        "verbosity": -1,           # Silenciar logs de entrenamiento
        "device": "cpu",
        "n_jobs": -1               # Usar todos los núcleos disponibles
    })
    return LGBMClassifier(**params)


In [ ]:
def build_catboost(best_params, random_state=RANDOM_SEED):
    """
    Construye un CatBoostClassifier con los mejores hiperparámetros de Optuna.
    Añade los parámetros fijos de configuración (función de pérdida, métrica, dispositivo).
    """
    params = best_params.copy()  # Evitar modificar el dict original
    params.update({
        "loss_function": "Logloss",  # Función de pérdida para clasificación binaria
        "eval_metric": "AUC",
        "verbose": 0,                 # Silenciar logs de entrenamiento
        "task_type": "CPU",
        "random_seed": random_state
    })
    return CatBoostClassifier(**params)


In [ ]:
def build_xgb(best_params, random_state=RANDOM_SEED):
    """
    Construye un XGBClassifier con los mejores hiperparámetros de Optuna.
    Añade los parámetros fijos de configuración (objetivo, métrica, semilla).
    """
    params = best_params.copy()  # Evitar modificar el dict original
    params.update({
        "objective": "binary:logistic",  # Clasificación binaria con salida probabilística
        "eval_metric": "auc",
        "random_state": random_state,
        "enable_categorical": False,      # Las categóricas ya están codificadas
        "n_jobs": -1
    })
    return XGBClassifier(**params)


### `generate_oof_and_test_preds`
Entrena el modelo en cada fold y genera:
- **OOF predictions**: predicciones sobre los datos de validación de cada fold (sin data leakage).
- **Test predictions**: promedio de las predicciones sobre test de todos los folds.

Estas predicciones OOF se usarán como features del metamodelo en el stacking.


In [ ]:
def generate_oof_and_test_preds(model, X, y, X_test, n_splits=3, random_state=RANDOM_SEED):
    """
    Entrena el modelo con validación cruzada y genera predicciones OOF y sobre test.

    Las predicciones OOF son honestas: cada observación se predice con un modelo
    que no la ha visto durante el entrenamiento (sin data leakage).

    Las predicciones sobre test son el promedio de los n_splits modelos entrenados.

    Retorna un diccionario con:
        oof_preds   : array con predicciones OOF para todo el conjunto de entrenamiento
        test_preds  : array con predicciones promediadas sobre el conjunto de test
        fold_scores : lista de AUC por fold
        oof_auc     : AUC global calculado sobre todas las predicciones OOF
        models      : lista de modelos entrenados (uno por fold)
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # Arrays para acumular predicciones
    oof_preds  = np.zeros(len(X))       # Una predicción por cada fila de train
    test_preds = np.zeros(len(X_test))  # Promedio de predicciones sobre test
    fold_scores = []
    trained_models = []

    # Detectar columnas categóricas para CatBoost
    cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        X_train_fold = X.iloc[train_idx].copy()
        X_valid_fold = X.iloc[valid_idx].copy()
        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        # Clonar el modelo para evitar que los folds compartan estado interno
        clf = clone(model)

        # Entrenar con manejo especial para CatBoost
        if isinstance(clf, CatBoostClassifier):
            clf.fit(X_train_fold, y_train_fold, cat_features=cat_features)
        else:
            clf.fit(X_train_fold, y_train_fold)

        # Predicciones OOF: sobre el fold de validación (no visto en entrenamiento)
        valid_pred = clf.predict_proba(X_valid_fold)[:, 1]
        oof_preds[valid_idx] = valid_pred

        # AUC del fold actual
        fold_score = roc_auc_score(y_valid_fold, valid_pred)
        fold_scores.append(fold_score)

        # Acumular predicciones sobre test (se promediarán al final)
        test_preds += clf.predict_proba(X_test)[:, 1] / n_splits
        trained_models.append(clf)

        print(f"Fold {fold+1} AUC: {fold_score:.6f}")

    # AUC global calculado sobre todas las predicciones OOF
    full_auc = roc_auc_score(y, oof_preds)
    print(f"OOF AUC: {full_auc:.6f}")

    return {
        "oof_preds":   oof_preds,
        "test_preds":  test_preds,
        "fold_scores": fold_scores,
        "oof_auc":     full_auc,
        "models":      trained_models
    }


### Stacking: `add_stack_features` y `train_meta_model`
El **stacking** combina las predicciones de los tres modelos base como nuevas variables
de entrada para un metamodelo (regresión logística).
- `add_stack_features`: añade features derivadas de las predicciones (media, máximo, diferencias entre modelos).
- `train_meta_model`: entrena la regresión logística sobre las predicciones OOF.


In [ ]:
def add_stack_features(df):
    """
    Enriquece el DataFrame de predicciones del nivel 1 con features derivadas.
    Estas features adicionales ayudan al metamodelo a capturar el acuerdo/desacuerdo
    entre los modelos base.

    Features generadas:
        mean_pred     : media de las tres predicciones (consenso general)
        max_pred      : predicción más alta (modelo más optimista)
        min_pred      : predicción más baja (modelo más conservador)
        std_pred      : desviación estándar (grado de desacuerdo entre modelos)
        lgbm_cat_diff : diferencia entre LGBM y CatBoost
        lgbm_xgb_diff : diferencia entre LGBM y XGBoost
        cat_xgb_diff  : diferencia entre CatBoost y XGBoost
    """
    df = df.copy()
    df["mean_pred"]     = df.mean(axis=1)                          # Consenso entre modelos
    df["max_pred"]      = df.max(axis=1)                           # Predicción más alta
    df["min_pred"]      = df.min(axis=1)                           # Predicción más baja
    df["std_pred"]      = df.std(axis=1)                           # Desacuerdo entre modelos
    df["lgbm_cat_diff"] = df["lgbm_pred"] - df["cat_pred"]        # Diferencia LGBM vs CatBoost
    df["lgbm_xgb_diff"] = df["lgbm_pred"] - df["xgb_pred"]        # Diferencia LGBM vs XGBoost
    df["cat_xgb_diff"]  = df["cat_pred"]  - df["xgb_pred"]        # Diferencia CatBoost vs XGBoost
    return df


In [ ]:
def train_meta_model(stack_train, y_train, stack_test, n_splits=3, random_state=RANDOM_SEED):
    """
    Entrena el metamodelo (regresión logística) sobre las predicciones OOF del nivel 1.
    Usa validación cruzada para generar predicciones OOF del propio metamodelo.

    Parámetros:
        stack_train : DataFrame con predicciones OOF de los modelos base (+ features derivadas)
        y_train     : variable objetivo del conjunto de entrenamiento
        stack_test  : DataFrame con predicciones sobre test de los modelos base
        n_splits    : número de folds para la validación cruzada del metamodelo
        random_state: semilla para reproducibilidad

    Retorna un diccionario con OOF, predicciones sobre test, scores por fold y AUC global.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_meta  = np.zeros(len(stack_train))  # Predicciones OOF del metamodelo
    test_meta = np.zeros(len(stack_test))   # Predicciones sobre test (promediadas)
    fold_scores = []
    meta_models = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(stack_train, y_train)):
        X_tr  = stack_train.iloc[train_idx]
        X_val = stack_train.iloc[valid_idx]
        y_tr  = y_train.iloc[train_idx]
        y_val = y_train.iloc[valid_idx]

        # Regresión logística como metamodelo: simple, interpretable y resistente al overfitting
        meta_model = LogisticRegression(C=1.0, max_iter=1000)
        meta_model.fit(X_tr, y_tr)

        # Predicciones OOF del metamodelo
        val_pred = meta_model.predict_proba(X_val)[:, 1]
        oof_meta[valid_idx] = val_pred

        fold_score = roc_auc_score(y_val, val_pred)
        fold_scores.append(fold_score)

        # Acumular predicciones sobre test (se promediarán)
        test_meta += meta_model.predict_proba(stack_test)[:, 1] / n_splits
        meta_models.append(meta_model)

        print(f"Meta fold {fold+1} AUC: {fold_score:.6f}")

    # AUC global del metamodelo sobre todas las predicciones OOF
    meta_auc = roc_auc_score(y_train, oof_meta)
    print(f"Meta OOF AUC: {meta_auc:.6f}")
    print(f"Meta CV AUC: {np.mean(fold_scores):.6f} +/- {np.std(fold_scores):.6f}")

    return {
        "oof_preds":   oof_meta,
        "test_preds":  test_meta,
        "fold_scores": fold_scores,
        "oof_auc":     meta_auc,
        "models":      meta_models
    }


### Funciones de visualización de resultados
- `extract_feature_importance` / `plot_feature_importance`: importancia de variables por modelo.
- `plot_confusion_matrix_with_pct`: matriz de confusión con porcentajes por fila.
- `plot_precision_recall`: curva Precisión-Recall y Average Precision.
- `plot_roc_curve_from_probs`: curva AUC-ROC.
- `evaluate_oof_predictions`: orquesta las tres visualizaciones anteriores para un modelo.


In [ ]:
def extract_feature_importance(model, feature_names):
    """
    Extrae la importancia de características de un modelo entrenado.
    Compatible con LightGBM, XGBoost, CatBoost y modelos sklearn con feature_importances_.

    Parámetros:
        model         : modelo entrenado
        feature_names : lista con los nombres de las features

    Retorna:
        DataFrame con columnas 'feature' e 'importance'
    """
    # Caso 1: modelos sklearn estándar y LightGBM (atributo feature_importances_)
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_

    # Caso 2: XGBoost (wrapper sklearn, acceso vía booster interno)
    elif hasattr(model, "get_booster"):
        booster = model.get_booster()
        score_dict = booster.get_score(importance_type="gain")  # Ganancia media por split
        importances = np.array([score_dict.get(f, 0.0) for f in feature_names])

    # Caso 3: CatBoost (método propio get_feature_importance)
    elif hasattr(model, "get_feature_importance"):
        importances = model.get_feature_importance()

    else:
        raise ValueError(f"El modelo {type(model)} no soporta extracción de importancia.")

    return pd.DataFrame({"feature": feature_names, "importance": importances})


In [ ]:
def plot_feature_importance(model_or_models, feature_names, top_n=20, title="Feature Importance"):
    """
    Visualiza la importancia de características para un modelo o lista de modelos (folds).
    Si se pasa una lista, promedia las importancias entre todos los modelos.

    Parámetros:
        model_or_models : modelo único o lista de modelos entrenados por fold
        feature_names   : lista con los nombres de las features
        top_n           : número de features más importantes a mostrar
        title           : título del gráfico
    """
    # Si se pasa una lista de modelos (uno por fold), promediar importancias
    if isinstance(model_or_models, list):
        fi_list = [extract_feature_importance(m, feature_names) for m in model_or_models]
        fi_df = pd.concat(fi_list, axis=0)
        fi_df = fi_df.groupby("feature", as_index=False)["importance"].mean()
    else:
        fi_df = extract_feature_importance(model_or_models, feature_names)

    # Seleccionar las top_n features y ordenar para el gráfico horizontal
    fi_df = fi_df.sort_values("importance", ascending=False).head(top_n)
    fi_df = fi_df.sort_values("importance", ascending=True)  # Orden ascendente para barh

    plt.figure(figsize=(10, max(6, top_n * 0.35)))
    plt.barh(fi_df["feature"], fi_df["importance"])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title(title)
    plt.tight_layout()
    plt.show()

    return fi_df.sort_values("importance", ascending=False)


In [ ]:
def plot_confusion_matrix_with_pct(y_true, y_prob, threshold=0.5, normalize="true", title="Confusion Matrix"):
    """
    Genera una matriz de confusión con anotaciones de conteo y porcentaje.

    Parámetros:
        y_true    : etiquetas reales
        y_prob    : probabilidades predichas
        threshold : umbral de clasificación (por defecto 0.5)
        normalize : 'true' → porcentajes por fila (recomendado)
                    'all'  → porcentajes globales
                    None   → solo conteos absolutos
        title     : título del gráfico
    """
    # Convertir probabilidades a etiquetas binarias según el umbral
    y_pred = (np.array(y_prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    # Calcular porcentajes según el modo de normalización
    if normalize == "true":
        cm_pct = cm / cm.sum(axis=1, keepdims=True)  # Porcentaje por fila (clase real)
    elif normalize == "all":
        cm_pct = cm / cm.sum()                        # Porcentaje global
    else:
        cm_pct = None

    # Construir anotaciones: conteo + porcentaje en cada celda
    annot = np.empty_like(cm).astype(str)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            count = cm[i, j]
            if cm_pct is not None:
                pct = cm_pct[i, j] * 100
                annot[i, j] = f"{count}\n{pct:.1f}%"
            else:
                annot[i, j] = str(count)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

    return cm


In [ ]:
def plot_precision_recall(y_true, y_prob, title="Precision-Recall Curve"):
    """
    Genera la curva Precisión-Recall y calcula el Average Precision (AP).
    Especialmente útil cuando hay desbalanceo de clases, ya que el AUC-ROC
    puede ser optimista en esos casos.

    Parámetros:
        y_true : etiquetas reales
        y_prob : probabilidades predichas
        title  : título del gráfico

    Retorna:
        Diccionario con arrays de precisión, recall, umbrales y el AP score.
    """
    # Calcular la curva Precisión-Recall para todos los umbrales posibles
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    # Average Precision: área bajo la curva P-R (resumen en un solo número)
    ap = average_precision_score(y_true, y_prob)

    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision, label=f"AP = {ap:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {"precision": precision, "recall": recall, "thresholds": thresholds, "average_precision": ap}


In [ ]:
def plot_roc_curve_from_probs(y_true, y_prob, title="ROC Curve"):
    """
    Genera la curva ROC y calcula el AUC (Area Under the Curve).
    La curva ROC muestra el trade-off entre la tasa de verdaderos positivos (TPR)
    y la tasa de falsos positivos (FPR) para todos los umbrales posibles.

    Parámetros:
        y_true : etiquetas reales
        y_prob : probabilidades predichas
        title  : título del gráfico

    Retorna:
        Diccionario con arrays de FPR, TPR, umbrales y el AUC score.
    """
    # Calcular la curva ROC para todos los umbrales posibles
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)

    # AUC: probabilidad de que el modelo clasifique mejor un positivo que un negativo
    auc = roc_auc_score(y_true, y_prob)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Clasificador aleatorio")  # Línea base
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {"fpr": fpr, "tpr": tpr, "thresholds": thresholds, "auc": auc}


In [ ]:
def evaluate_oof_predictions(y_true, y_prob, threshold=0.5, model_name="Model"):
    """
    Orquesta la evaluación completa de un modelo sobre sus predicciones OOF.
    Genera tres visualizaciones: matriz de confusión, curva P-R y curva ROC.

    Parámetros:
        y_true     : etiquetas reales del conjunto de entrenamiento
        y_prob     : predicciones OOF del modelo
        threshold  : umbral de clasificación para la matriz de confusión
        model_name : nombre del modelo (para los títulos de los gráficos)
    """
    print(f"{model_name} — evaluación con umbral={threshold:.2f}")

    # Matriz de confusión con porcentajes por fila
    cm = plot_confusion_matrix_with_pct(
        y_true=y_true, y_prob=y_prob,
        threshold=threshold,
        title=f"{model_name} Confusion Matrix"
    )

    # Curva Precisión-Recall
    pr_info = plot_precision_recall(
        y_true=y_true, y_prob=y_prob,
        title=f"{model_name} Precision-Recall Curve"
    )

    # Curva ROC
    roc_info = plot_roc_curve_from_probs(
        y_true=y_true, y_prob=y_prob,
        title=f"{model_name} ROC Curve"
    )

    return {"confusion_matrix": cm, "pr_info": pr_info, "roc_info": roc_info}


## 4. Carga de datos

Cargamos el dataset original de IBM desde el directorio local y lo dividimos en:
- `train`: 80% de los datos, con la etiqueta `Churn` y la bandera numérica `churn_flag`.
- `test`: 20% restante, sin etiqueta (simula el conjunto de evaluación).
- `sub`: plantilla de submission con columnas `id` y `prediction` (inicializada a 0).
- `original_data`: copia del dataset completo, usada como referencia para el target encoding.

La división es estratificada para mantener la proporción de churn en ambos conjuntos.


In [ ]:
from sklearn.model_selection import train_test_split

# Cargar el dataset original de IBM Telco Customer Churn
original_data = pd.read_csv("./data/telco-churn.csv")

# TotalCharges viene como string en el CSV original; convertir a numérico
# Los valores no convertibles (espacios en blanco) se marcan como NaN
original_data['TotalCharges'] = pd.to_numeric(original_data['TotalCharges'], errors='coerce')

# Eliminar las filas con TotalCharges nulo (clientes con tenure=0, sin cargos registrados)
original_data.dropna(subset=['TotalCharges'], inplace=True)

# Crear la variable objetivo numérica: 1 = churn, 0 = no churn
original_data['churn_flag'] = np.where(original_data['Churn'] == 'No', 0, 1)

# Añadir columna de identificador único para el fichero de submission
original_data.insert(0, 'id', range(len(original_data)))

# División estratificada 80/20: mantiene la proporción de churn en ambos conjuntos
train, test = train_test_split(
    original_data,
    test_size=0.2,
    stratify=original_data['churn_flag'],
    random_state=RANDOM_SEED
)
train = train.reset_index(drop=True)
test  = test.reset_index(drop=True)

# Persistir los conjuntos en disco para reproducibilidad
train.to_csv("./data/train.csv", index=False)
test.drop(columns=['Churn', 'churn_flag']).to_csv("./data/test.csv", index=False)

# Crear plantilla de submission con predicciones inicializadas a 0
sub = pd.DataFrame({'id': test['id'], 'prediction': 0})
sub.to_csv("./data/sample_submission.csv", index=False)

# Eliminar la etiqueta del conjunto de test en memoria (simula el escenario real)
test = test.drop(columns=['Churn', 'churn_flag'])

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Tasa de churn en train: {train['churn_flag'].mean():.3f}")


## 5. Descripción del Dataset

El dataset utilizado es el **IBM Telco Customer Churn**, que recoge información sobre los clientes de una empresa de telecomunicaciones ficticia.
Contiene **7.043 registros** y **21 columnas** originales. Cada fila representa un cliente único.

El objetivo es predecir la columna `Churn`, que indica si el cliente abandonó el servicio en el último mes.

---

### 5.1 Identificador

| Variable | Tipo | Descripción |
|---|---|---|
| `customerID` | string | Identificador único del cliente (ej. `7590-VHVEG`). No aporta información predictiva. |

---

### 5.2 Variables demográficas

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `gender` | categórica | `Male`, `Female` | Género del cliente. |
| `SeniorCitizen` | binaria | `0`, `1` | Indica si el cliente tiene 65 años o más (`1` = sí). |
| `Partner` | categórica | `Yes`, `No` | Indica si el cliente tiene pareja. |
| `Dependents` | categórica | `Yes`, `No` | Indica si el cliente tiene personas a su cargo (hijos, padres, etc.). |

---

### 5.3 Variables de antigüedad y contrato

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `tenure` | numérica continua | 0 – 72 | Número de meses que el cliente lleva con la compañía. |
| `Contract` | categórica | `Month-to-month`, `One year`, `Two year` | Tipo de contrato. Los contratos mensuales tienen mayor tasa de churn. |
| `PaperlessBilling` | categórica | `Yes`, `No` | Indica si el cliente recibe la factura en formato electrónico. |
| `PaymentMethod` | categórica | `Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)` | Método de pago utilizado. |

---

### 5.4 Variables de servicios de telefonía

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `PhoneService` | categórica | `Yes`, `No` | Indica si el cliente tiene contratado el servicio de telefonía. |
| `MultipleLines` | categórica | `Yes`, `No`, `No phone service` | Indica si el cliente tiene más de una línea telefónica. El valor `No phone service` aparece cuando `PhoneService = No`. |

---

### 5.5 Variables de servicios de internet

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `InternetService` | categórica | `DSL`, `Fiber optic`, `No` | Tipo de conexión a internet contratada. `Fiber optic` tiene mayor coste y mayor tasa de churn. |
| `OnlineSecurity` | categórica | `Yes`, `No`, `No internet service` | Servicio de seguridad online (antivirus, protección frente a amenazas). |
| `OnlineBackup` | categórica | `Yes`, `No`, `No internet service` | Servicio de copia de seguridad en la nube. |
| `DeviceProtection` | categórica | `Yes`, `No`, `No internet service` | Seguro de protección para los dispositivos del cliente. |
| `TechSupport` | categórica | `Yes`, `No`, `No internet service` | Servicio de soporte técnico prioritario. |
| `StreamingTV` | categórica | `Yes`, `No`, `No internet service` | Servicio de streaming de televisión a través de internet. |
| `StreamingMovies` | categórica | `Yes`, `No`, `No internet service` | Servicio de streaming de películas a través de internet. |

> Los valores `No internet service` en los servicios anteriores indican que el cliente no tiene contratado ningún tipo de internet (`InternetService = No`).

---

### 5.6 Variables de facturación

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `MonthlyCharges` | numérica continua | 18.25 – 118.75 | Importe facturado al cliente en el mes actual (en USD). |
| `TotalCharges` | numérica continua | 18.8 – 8684.8 | Importe total facturado al cliente desde que se dio de alta. Viene como `string` en el CSV original y contiene espacios en blanco para clientes con `tenure = 0`. |

---

### 5.7 Variable objetivo

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `Churn` | categórica | `Yes`, `No` | Indica si el cliente abandonó el servicio en el último mes. Es la variable que el modelo debe predecir. |
| `churn_flag` | binaria (derivada) | `0`, `1` | Versión numérica de `Churn` creada durante la carga de datos (`1` = churn, `0` = no churn). |

---

### 5.8 Notas sobre calidad del dato

- **`TotalCharges`** contiene 11 valores vacíos (espacios en blanco) correspondientes a clientes con `tenure = 0`, es decir, clientes que se dieron de alta pero aún no han recibido ninguna factura. Estos registros se eliminan durante la carga.
- **`SeniorCitizen`** es la única variable demográfica codificada como entero (`0`/`1`) en lugar de `Yes`/`No`.
- El dataset presenta un **desbalanceo de clases**: aproximadamente el 26% de los clientes son churn frente al 74% que no lo son.


## 6. Análisis Exploratorio de Datos (EDA)

El objetivo del EDA es entender la estructura del dataset antes de modelar:
- ¿Cuántas filas y columnas hay?
- ¿Hay valores nulos o tipos incorrectos?
- ¿Cómo se distribuye la variable objetivo? ¿Hay desbalanceo?
- ¿Qué variables están más relacionadas con el churn?


### 5.1 Resumen del dataset de entrenamiento


In [ ]:
# Mostrar resumen completo del conjunto de entrenamiento
describe_dataset(train)

### 5.2 Resumen del dataset original (IBM)
Comparamos las estadísticas del dataset sintético de Kaggle con el dataset real de IBM.


In [ ]:
# Mostrar resumen del dataset original de IBM para comparación
describe_dataset(original_data)

### 5.3 Distribución de la variable objetivo
Analizamos el desbalanceo de clases. Un desbalanceo severo puede requerir técnicas
como sobremuestreo (SMOTE) o ajuste del umbral de clasificación.


In [ ]:
# Visualizar la distribución de la variable objetivo (churn vs no churn)
plot_binary_feature(train, TARGET)

### 5.4 Distribución de variables categóricas
Visualizamos la distribución de cada variable categórica para detectar
categorías dominantes o poco representadas.


In [ ]:
# Iterar sobre todas las variables categóricas y visualizar su distribución
for col in CATEGORY_COLUMNS:
    plot_categorical_distribution(train, col)

### 5.5 Variables categóricas vs. churn
Para cada variable categórica, calculamos la tasa de churn por categoría.
Esto nos permite identificar qué valores están más asociados con la baja del cliente.


In [ ]:
# Calcular y visualizar la tasa de churn para cada categoría de cada variable
for col in CATEGORY_COLUMNS:
    categorical_vs_target(train, col, TARGET)

### 5.6 Distribución de variables numéricas continuas
Analizamos la forma de la distribución (asimetría, curtosis) y si se aleja de la normalidad.
Esto es relevante para decidir si aplicar transformaciones antes del modelado.


In [ ]:
# Analizar la distribución de las variables numéricas continuas
# Retorna un DataFrame con asimetría, curtosis, p-valor de normalidad e interpretación
train_summary = analyze_numeric_distribution(train, NUMERIC_CONTINUOUS)
train_summary

### 5.7 Variables numéricas vs. churn
Comparamos la distribución de cada variable numérica entre clientes que se dan de baja
y los que no. Incluye test estadístico (Welch t-test) y tamaño del efecto (Cohen's d).


In [ ]:
# Comparar la distribución de cada variable numérica entre clientes con y sin churn
for col in NUMERIC_CONTINUOUS:
    numeric_vs_target(train, col, TARGET)

### 5.8 Matriz de correlación
Visualizamos las correlaciones entre variables numéricas.
Correlaciones altas entre predictores pueden indicar multicolinealidad.


In [ ]:
# Generar el mapa de calor de correlaciones entre variables numéricas
plot_correlation_matrix(train)

## 7. Ingeniería de características

Creamos nuevas variables a partir de las existentes para enriquecer la información
disponible para los modelos. Las nuevas features capturan relaciones que las variables
originales no expresan directamente:

| Feature nueva | Descripción |
|---|---|
| `auto_pay` | 1 si el cliente paga de forma automática |
| `digital_customer` | 1 si tiene facturación electrónica |
| `payment_risk` | Suma de indicadores de riesgo de pago |
| `household_size_proxy` | Estimación del tamaño del hogar |
| `contract_length` | Duración del contrato en meses |
| `contract_tenure_ratio` | Proporción del contrato completada |
| `avg_monthly_charges` | Cargo mensual medio histórico |
| `charge_ratio` | Ratio entre cargo actual y medio |
| `num_services` / `num_addons` | Número de servicios contratados |
| `cost_per_service` | Coste por servicio contratado |
| `wrong_package_flag` | Cliente con pocos servicios pero tarifa alta |

Además, aplicamos **target encoding** sobre las variables categóricas usando el dataset
original de IBM como referencia, evitando así el data leakage.


### `feature_engineering`
Genera todas las nuevas variables a partir de las columnas originales del dataset.
Se aplica tanto al conjunto de entrenamiento como al de test para garantizar consistencia.


In [ ]:
def feature_engineering(df):
    """
    Genera nuevas variables a partir de las columnas originales del dataset.
    Captura relaciones de negocio que los modelos no pueden inferir directamente.
    """
    # Servicios principales y complementarios
    core   = ["PhoneService", "InternetService"]
    addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]
    services = core + addons

    # ── Variables de comportamiento de pago ───────────────────────────────
    # 1 si el cliente paga de forma automática (menor riesgo de impago)
    df["auto_pay"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    # 1 si el cliente tiene facturación electrónica
    df["digital_customer"] = (df["PaperlessBilling"] == "Yes").astype(int)

    # Suma de indicadores de riesgo: pago no automático + facturación en papel
    df["payment_risk"] = ((df["auto_pay"] == 0).astype(int) +
                           (df["digital_customer"] == 0).astype(int))

    # ── Variables demográficas y de hogar ─────────────────────────────────
    # Proxy del tamaño del hogar: cliente + pareja + dependientes
    df["household_size_proxy"] = (1 +
        (df["Partner"] == "Yes").astype(int) +
        (df["Dependents"] == "Yes").astype(int))

    # ── Variables de contrato y antigüedad ────────────────────────────────
    # Duración del contrato en meses
    df["contract_length"] = df["Contract"].map(
        {"Month-to-month": 1, "One year": 12, "Two year": 24})

    # Proporción del contrato completada (0 = inicio, >1 = renovado)
    df["contract_tenure_ratio"] = df["tenure"] / (df["contract_length"] + 1)

    # ── Variables de facturación ──────────────────────────────────────────
    # Cargo mensual medio histórico (TotalCharges / meses)
    df["avg_monthly_charges"] = df["TotalCharges"] / (df["tenure"] + 1)

    # Ratio entre el cargo actual y el medio histórico (detecta subidas recientes)
    df["charge_ratio"] = df["MonthlyCharges"] / (df["avg_monthly_charges"] + 1)

    # Gasto mensual por persona del hogar
    df["spending_per_person"] = df["MonthlyCharges"] / (df["household_size_proxy"] + 1)

    # Total esperado si el cargo mensual se hubiera mantenido constante
    df["expected_total"] = df["MonthlyCharges"] * df["tenure"]

    # Desviación entre el total real y el esperado (detecta cambios de tarifa)
    df["billing_deviation"] = df["TotalCharges"] / (df["expected_total"] + 1)

    # ── Variables de servicios contratados ────────────────────────────────
    # Número total de servicios activos
    df["num_services"] = df[services].apply(lambda x: (x == "Yes").sum(), axis=1)

    # Número de servicios complementarios (add-ons)
    df["num_addons"] = df[addons].apply(lambda x: (x == "Yes").sum(), axis=1)

    # Proporción de add-ons sobre el total de servicios
    df["addons_to_total_ratio"] = df["num_addons"] / (df["num_services"] + 1)

    # 1 si el cliente tiene los tres servicios de protección contratados
    df["has_protection_bundle"] = (
        (df["OnlineSecurity"] == "Yes") &
        (df["OnlineBackup"] == "Yes") &
        (df["DeviceProtection"] == "Yes")
    ).astype(int)

    # Coste mensual por servicio contratado
    df["cost_per_service"] = df["MonthlyCharges"] / (df["num_services"] + 1)

    # Flag: cliente con pocos servicios pero tarifa superior a la mediana (posible mal encaje)
    df["wrong_package_flag"] = (
        (df["num_services"] <= 2) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    return df


### `target_encode_from_orig`
Aplica target encoding sobre las variables categóricas usando el dataset original de IBM
como fuente de estadísticas, evitando data leakage en train y test.
Para cada categoría genera: media, mediana, desviación estándar, percentiles 10/90,
conteo, diferencia con la media global y ranking percentil.


In [ ]:
def target_encode_from_orig(train, test, orig, cat_cols, target):
    """
    Aplica target encoding usando el dataset original como referencia.
    Evita data leakage: las estadísticas se calculan sobre 'orig', no sobre 'train'.

    Para cada variable categórica genera 8 features numéricas:
        _te_mean      : tasa media de churn por categoría
        _te_median    : tasa mediana de churn por categoría
        _te_std       : variabilidad de la tasa de churn
        _te_q10       : percentil 10 de la tasa de churn
        _te_q90       : percentil 90 de la tasa de churn
        _te_count     : número de observaciones por categoría
        _te_mean_diff : diferencia entre la media de la categoría y la media global
        _te_rank_pct  : ranking percentil de la categoría por tasa de churn
    """
    train = train.copy()
    test  = test.copy()

    # Estadísticas globales del target (para imputar categorías no vistas)
    global_mean = orig[target].mean()
    global_std  = orig[target].std()

    for col in cat_cols:
        # Calcular estadísticas del target por categoría sobre el dataset original
        stats = orig.groupby(col)[target].agg(
            mean="mean",
            median="median",
            std="std",
            q10=lambda x: x.quantile(0.1),
            q90=lambda x: x.quantile(0.9),
            count="count"
        ).reset_index()

        # Features adicionales derivadas de las estadísticas
        stats[f"{col}_te_mean_diff"] = stats["mean"] - global_mean  # Desviación respecto a la media global
        stats[f"{col}_te_rank_pct"]  = stats["mean"].rank(pct=True)  # Ranking percentil

        # Renombrar columnas con prefijo de la variable original
        stats = stats.rename(columns={
            "mean":   f"{col}_te_mean",
            "median": f"{col}_te_median",
            "std":    f"{col}_te_std",
            "q10":    f"{col}_te_q10",
            "q90":    f"{col}_te_q90",
            "count":  f"{col}_te_count"
        })

        # Unir las estadísticas a train y test mediante merge
        train = train.merge(stats, on=col, how="left")
        test  = test.merge(stats, on=col, how="left")

        # Imputar categorías no vistas con valores por defecto
        fill_map = {
            f"{col}_te_mean":      global_mean,
            f"{col}_te_median":    global_mean,
            f"{col}_te_std":       global_std,
            f"{col}_te_q10":       global_mean,
            f"{col}_te_q90":       global_mean,
            f"{col}_te_count":     0,
            f"{col}_te_mean_diff": 0,
            f"{col}_te_rank_pct":  0.5,
        }
        for c, val in fill_map.items():
            train[c] = train[c].fillna(val).astype("float32")
            test[c]  = test[c].fillna(val).astype("float32")

    # Eliminar las columnas categóricas originales (ya codificadas)
    train.drop(columns=cat_cols, inplace=True)
    test.drop(columns=cat_cols, inplace=True)

    return train, test


In [ ]:
# Aplicar ingeniería de características a train y test
train = feature_engineering(train)
test  = feature_engineering(test)

# Aplicar target encoding usando el dataset original como referencia
# Esto evita data leakage: las estadísticas no se calculan sobre train
train, test = target_encode_from_orig(
    train=train,
    test=test,
    orig=original_data,
    cat_cols=CATEGORY_COLUMNS,
    target=TARGET
)

## Arquitectura del Pipeline

<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 1100 920' font-family='Segoe UI, Arial, sans-serif'>
  <rect width='1100' height='920' fill='#F7F9FC' rx='12'/>
  <text x='550' y='42' text-anchor='middle' font-size='20' font-weight='bold' fill='#1A1A2E'>Arquitectura del Pipeline de Predicción de Churn</text>
  <text x='550' y='64' text-anchor='middle' font-size='13' fill='#555'>Stacking de dos niveles · LightGBM · CatBoost · XGBoost · Regresión Logística</text>
  <rect x='30' y='85' width='1040' height='110' rx='10' fill='#EEF2FF' stroke='#C7D2FE' stroke-width='1.5'/>
  <text x='55' y='107' font-size='11' font-weight='bold' fill='#4338CA'>DATOS DE ENTRADA</text>
  <rect x='55' y='115' width='180' height='64' rx='8' fill='#4F46E5'/>
  <text x='145' y='140' text-anchor='middle' font-size='12' font-weight='bold' fill='white'>Dataset Original</text>
  <text x='145' y='156' text-anchor='middle' font-size='10.5' fill='#C7D2FE'>IBM Telco Churn</text>
  <text x='145' y='170' text-anchor='middle' font-size='10' fill='#A5B4FC'>7.043 clientes · 21 variables</text>
  <line x1='235' y1='147' x2='283' y2='147' stroke='#6366F1' stroke-width='2' marker-end='url(#ab)'/>
  <rect x='285' y='115' width='160' height='64' rx='8' fill='#6366F1'/>
  <text x='365' y='137' text-anchor='middle' font-size='12' font-weight='bold' fill='white'>Train / Test Split</text>
  <text x='365' y='153' text-anchor='middle' font-size='10.5' fill='#C7D2FE'>Estratificado 80 / 20</text>
  <text x='365' y='168' text-anchor='middle' font-size='10' fill='#A5B4FC'>random_state = 27</text>
  <line x1='445' y1='147' x2='493' y2='147' stroke='#6366F1' stroke-width='2' marker-end='url(#ab)'/>
  <rect x='495' y='115' width='175' height='64' rx='8' fill='#7C3AED'/>
  <text x='582' y='137' text-anchor='middle' font-size='12' font-weight='bold' fill='white'>Feature Engineering</text>
  <text x='582' y='153' text-anchor='middle' font-size='10.5' fill='#DDD6FE'>11 nuevas variables</text>
  <text x='582' y='168' text-anchor='middle' font-size='10' fill='#C4B5FD'>auto_pay · tenure_ratio · …</text>
  <line x1='670' y1='147' x2='718' y2='147' stroke='#6366F1' stroke-width='2' marker-end='url(#ab)'/>
  <rect x='720' y='115' width='175' height='64' rx='8' fill='#9333EA'/>
  <text x='807' y='137' text-anchor='middle' font-size='12' font-weight='bold' fill='white'>Target Encoding</text>
  <text x='807' y='153' text-anchor='middle' font-size='10.5' fill='#E9D5FF'>16 variables categóricas</text>
  <text x='807' y='168' text-anchor='middle' font-size='10' fill='#D8B4FE'>8 stats · sin leakage</text>
  <line x1='895' y1='147' x2='943' y2='147' stroke='#6366F1' stroke-width='2' marker-end='url(#ab)'/>
  <rect x='945' y='115' width='110' height='64' rx='8' fill='#A855F7'/>
  <text x='1000' y='143' text-anchor='middle' font-size='12' font-weight='bold' fill='white'>Features</text>
  <text x='1000' y='161' text-anchor='middle' font-size='10' fill='#E9D5FF'>listas para modelado</text>
  <rect x='30' y='220' width='1040' height='310' rx='10' fill='#F0FDF4' stroke='#BBF7D0' stroke-width='1.5'/>
  <text x='55' y='242' font-size='11' font-weight='bold' fill='#15803D'>NIVEL 1 — MODELOS BASE  (Validación cruzada estratificada · 3 folds)</text>
  <line x1='1000' y1='179' x2='1000' y2='210' stroke='#A855F7' stroke-width='1.5'/>
  <line x1='220' y1='210' x2='880' y2='210' stroke='#A855F7' stroke-width='1.5'/>
  <line x1='220' y1='210' x2='220' y2='253' stroke='#A855F7' stroke-width='1.5' marker-end='url(#ap)'/>
  <line x1='550' y1='210' x2='550' y2='253' stroke='#A855F7' stroke-width='1.5' marker-end='url(#ap)'/>
  <line x1='880' y1='210' x2='880' y2='253' stroke='#A855F7' stroke-width='1.5' marker-end='url(#ap)'/>
  <rect x='60' y='255' width='320' height='255' rx='10' fill='white' stroke='#86EFAC' stroke-width='2'/>
  <rect x='60' y='255' width='320' height='36' rx='10' fill='#16A34A'/>
  <rect x='60' y='279' width='320' height='12' fill='#16A34A'/>
  <text x='220' y='279' text-anchor='middle' font-size='13' font-weight='bold' fill='white'>LightGBM</text>
  <text x='80' y='315' font-size='10.5' fill='#166534' font-weight='bold'>Optimización (Optuna · 5 trials)</text>
  <rect x='80' y='322' width='280' height='58' rx='6' fill='#F0FDF4' stroke='#BBF7D0'/>
  <text x='92' y='338' font-size='10' fill='#374151'>n_estimators: 140–800  ·  num_leaves: 200–4000</text>
  <text x='92' y='354' font-size='10' fill='#374151'>learning_rate: 0.01–0.1 (log)</text>
  <text x='92' y='370' font-size='10' fill='#374151'>reg_alpha / reg_lambda: L1 + L2</text>
  <rect x='80' y='392' width='280' height='44' rx='6' fill='#F0FDF4' stroke='#BBF7D0'/>
  <text x='92' y='408' font-size='10' fill='#374151'>objective: binary  ·  metric: auc</text>
  <text x='92' y='424' font-size='10' fill='#374151'>boosting: gbdt  ·  device: cpu  ·  n_jobs: -1</text>
  <rect x='80' y='448' width='280' height='34' rx='6' fill='#DCFCE7' stroke='#86EFAC'/>
  <text x='220' y='465' text-anchor='middle' font-size='10.5' fill='#15803D' font-weight='bold'>OOF preds + Test preds → lgbm_pred</text>
  <rect x='390' y='255' width='320' height='255' rx='10' fill='white' stroke='#FCA5A5' stroke-width='2'/>
  <rect x='390' y='255' width='320' height='36' rx='10' fill='#DC2626'/>
  <rect x='390' y='279' width='320' height='12' fill='#DC2626'/>
  <text x='550' y='279' text-anchor='middle' font-size='13' font-weight='bold' fill='white'>CatBoost</text>
  <text x='410' y='315' font-size='10.5' fill='#991B1B' font-weight='bold'>Optimización (Optuna · 5 trials)</text>
  <rect x='410' y='322' width='280' height='58' rx='6' fill='#FFF5F5' stroke='#FCA5A5'/>
  <text x='422' y='338' font-size='10' fill='#374151'>iterations: 150–800  ·  depth: 4–16</text>
  <text x='422' y='354' font-size='10' fill='#374151'>l2_leaf_reg: 1e-3–10 (log)</text>
  <text x='422' y='370' font-size='10' fill='#374151'>bagging_temperature: 0–5</text>
  <rect x='410' y='392' width='280' height='44' rx='6' fill='#FFF5F5' stroke='#FCA5A5'/>
  <text x='422' y='408' font-size='10' fill='#374151'>loss_function: Logloss  ·  eval: AUC</text>
  <text x='422' y='424' font-size='10' fill='#374151'>task_type: CPU  ·  categóricas nativas</text>
  <rect x='410' y='448' width='280' height='34' rx='6' fill='#FEE2E2' stroke='#FCA5A5'/>
  <text x='550' y='465' text-anchor='middle' font-size='10.5' fill='#DC2626' font-weight='bold'>OOF preds + Test preds → cat_pred</text>
  <rect x='720' y='255' width='320' height='255' rx='10' fill='white' stroke='#FCD34D' stroke-width='2'/>
  <rect x='720' y='255' width='320' height='36' rx='10' fill='#D97706'/>
  <rect x='720' y='279' width='320' height='12' fill='#D97706'/>
  <text x='880' y='279' text-anchor='middle' font-size='13' font-weight='bold' fill='white'>XGBoost</text>
  <text x='740' y='315' font-size='10.5' fill='#92400E' font-weight='bold'>Optimización (Optuna · 5 trials)</text>
  <rect x='740' y='322' width='280' height='58' rx='6' fill='#FFFBEB' stroke='#FCD34D'/>
  <text x='752' y='338' font-size='10' fill='#374151'>n_estimators: 200–400  ·  max_depth: 3–16</text>
  <text x='752' y='354' font-size='10' fill='#374151'>gamma: 1e-4–10 (log)</text>
  <text x='752' y='370' font-size='10' fill='#374151'>min_child_weight: 1–20</text>
  <rect x='740' y='392' width='280' height='44' rx='6' fill='#FFFBEB' stroke='#FCD34D'/>
  <text x='752' y='408' font-size='10' fill='#374151'>objective: binary:logistic  ·  eval: auc</text>
  <text x='752' y='424' font-size='10' fill='#374151'>n_jobs: -1  ·  enable_categorical: False</text>
  <rect x='740' y='448' width='280' height='34' rx='6' fill='#FEF3C7' stroke='#FCD34D'/>
  <text x='880' y='465' text-anchor='middle' font-size='10.5' fill='#D97706' font-weight='bold'>OOF preds + Test preds → xgb_pred</text>
  <rect x='30' y='550' width='1040' height='130' rx='10' fill='#FFF7ED' stroke='#FED7AA' stroke-width='1.5'/>
  <text x='55' y='572' font-size='11' font-weight='bold' fill='#C2410C'>STACKING — CONSTRUCCIÓN DEL CONJUNTO DE NIVEL 2</text>
  <line x1='220' y1='510' x2='220' y2='540' stroke='#6B7280' stroke-width='1.5'/>
  <line x1='550' y1='510' x2='550' y2='540' stroke='#6B7280' stroke-width='1.5'/>
  <line x1='880' y1='510' x2='880' y2='540' stroke='#6B7280' stroke-width='1.5'/>
  <line x1='220' y1='540' x2='880' y2='540' stroke='#6B7280' stroke-width='1.5'/>
  <line x1='550' y1='540' x2='550' y2='553' stroke='#6B7280' stroke-width='1.5' marker-end='url(#ag)'/>
  <rect x='80' y='582' width='430' height='82' rx='8' fill='white' stroke='#FDBA74' stroke-width='1.5'/>
  <text x='295' y='601' text-anchor='middle' font-size='11' font-weight='bold' fill='#C2410C'>stack_train — Predicciones OOF concatenadas</text>
  <text x='100' y='620' font-size='10' fill='#374151'>Base: lgbm_pred · cat_pred · xgb_pred</text>
  <text x='100' y='635' font-size='10' fill='#374151'>Derivadas: mean · max · min · std</text>
  <text x='100' y='650' font-size='10' fill='#374151'>Diferencias: lgbm−cat · lgbm−xgb · cat−xgb</text>
  <rect x='530' y='582' width='510' height='82' rx='8' fill='white' stroke='#FDBA74' stroke-width='1.5'/>
  <text x='785' y='601' text-anchor='middle' font-size='11' font-weight='bold' fill='#C2410C'>stack_test — Predicciones sobre Test</text>
  <text x='550' y='620' font-size='10' fill='#374151'>Promedio de los 3 folds por modelo base</text>
  <text x='550' y='635' font-size='10' fill='#374151'>Mismas features derivadas aplicadas</text>
  <text x='550' y='650' font-size='10' fill='#374151'>Sin data leakage: stats calculadas sobre original_data</text>
  <rect x='30' y='700' width='1040' height='130' rx='10' fill='#EFF6FF' stroke='#BFDBFE' stroke-width='1.5'/>
  <text x='55' y='722' font-size='11' font-weight='bold' fill='#1D4ED8'>NIVEL 2 — METAMODELO</text>
  <line x1='295' y1='664' x2='295' y2='693' stroke='#F97316' stroke-width='1.5' marker-end='url(#ao)'/>
  <line x1='785' y1='664' x2='785' y2='693' stroke='#F97316' stroke-width='1.5' marker-end='url(#ao)'/>
  <rect x='80' y='730' width='430' height='82' rx='8' fill='#1D4ED8'/>
  <text x='295' y='754' text-anchor='middle' font-size='13' font-weight='bold' fill='white'>Regresión Logística (Metamodelo)</text>
  <text x='295' y='772' text-anchor='middle' font-size='10.5' fill='#BFDBFE'>Entrenada sobre stack_train · C=1.0 · max_iter=1000</text>
  <text x='295' y='788' text-anchor='middle' font-size='10.5' fill='#BFDBFE'>Validación cruzada estratificada · 3 folds</text>
  <text x='295' y='804' text-anchor='middle' font-size='10' fill='#93C5FD'>Simple · interpretable · resistente al overfitting</text>
  <rect x='530' y='730' width='510' height='82' rx='8' fill='#2563EB'/>
  <text x='785' y='754' text-anchor='middle' font-size='13' font-weight='bold' fill='white'>Predicciones Finales</text>
  <text x='785' y='772' text-anchor='middle' font-size='10.5' fill='#BFDBFE'>meta_res[test_preds] → submission.csv</text>
  <text x='785' y='788' text-anchor='middle' font-size='10.5' fill='#BFDBFE'>Probabilidad de churn por cliente [0, 1]</text>
  <text x='785' y='804' text-anchor='middle' font-size='10' fill='#93C5FD'>Métrica: AUC-ROC · Average Precision</text>
  <rect x='30' y='850' width='1040' height='52' rx='10' fill='#F8FAFC' stroke='#CBD5E1' stroke-width='1.5'/>
  <text x='55' y='870' font-size='11' font-weight='bold' fill='#475569'>EVALUACIÓN (sobre predicciones OOF — sin data leakage)</text>
  <text x='180' y='890' font-size='10.5' fill='#64748B'>AUC-ROC</text>
  <text x='340' y='890' font-size='10.5' fill='#64748B'>Curva Precisión-Recall · AP</text>
  <text x='590' y='890' font-size='10.5' fill='#64748B'>Matriz de confusión (umbral 0.5)</text>
  <text x='850' y='890' font-size='10.5' fill='#64748B'>Feature Importance</text>
  <line x1='295' y1='812' x2='295' y2='843' stroke='#3B82F6' stroke-width='1.5' marker-end='url(#ab2)'/>
  <line x1='785' y1='812' x2='785' y2='843' stroke='#3B82F6' stroke-width='1.5' marker-end='url(#ab2)'/>
  <defs>
    <marker id='ab'  markerWidth='8' markerHeight='8' refX='6' refY='3' orient='auto'><path d='M0,0 L0,6 L8,3 z' fill='#6366F1'/></marker>
    <marker id='ap'  markerWidth='8' markerHeight='8' refX='6' refY='3' orient='auto'><path d='M0,0 L0,6 L8,3 z' fill='#A855F7'/></marker>
    <marker id='ag'  markerWidth='8' markerHeight='8' refX='6' refY='3' orient='auto'><path d='M0,0 L0,6 L8,3 z' fill='#6B7280'/></marker>
    <marker id='ao'  markerWidth='8' markerHeight='8' refX='6' refY='3' orient='auto'><path d='M0,0 L0,6 L8,3 z' fill='#F97316'/></marker>
    <marker id='ab2' markerWidth='8' markerHeight='8' refX='6' refY='3' orient='auto'><path d='M0,0 L0,6 L8,3 z' fill='#3B82F6'/></marker>
  </defs>
</svg>


## 8. Entrenamiento y optimización de modelos

Seguimos una arquitectura de **stacking en dos niveles**:

**Nivel 1 – Modelos base:**
- LightGBM (LGBM)
- CatBoost
- XGBoost

Cada modelo se optimiza con **Optuna** (búsqueda bayesiana de hiperparámetros)
y se evalúa con validación cruzada estratificada de 3 folds.

**Nivel 2 – Metamodelo:**
- Regresión logística entrenada sobre las predicciones OOF de los modelos base.

### Definición de features de entrenamiento


In [ ]:
# Columnas a excluir del entrenamiento: identificadores y la variable objetivo
DROP_COLS = ["id", "customerID", 'Churn', "churn_flag"]

# Features que usarán los modelos: todas las columnas excepto las excluidas
MODEL_FEATURES = [col for col in train.columns if col not in DROP_COLS]

### 7.1 Optimización de hiperparámetros con Optuna
Lanzamos la búsqueda de hiperparámetros para los tres modelos base.
Cada `study` guarda el historial de trials y los mejores parámetros encontrados.


In [ ]:
# Optimizar hiperparámetros de LightGBM (5 trials)
print("Optimizando LightGBM...")
lgbm_study = tune_model(objective_lgbm, train[MODEL_FEATURES], train[TARGET],
                        n_trials=5, study_name="lgbm")

# Optimizar hiperparámetros de CatBoost (5 trials)
print("Optimizando CatBoost...")
cat_study  = tune_model(objective_catboost, train[MODEL_FEATURES], train[TARGET],
                        n_trials=5, study_name="catboost")

# Optimizar hiperparámetros de XGBoost (5 trials, más lento)
print("Optimizando XGBoost...")
xgb_study  = tune_model(objective_xgb, train[MODEL_FEATURES], train[TARGET],
                        n_trials=5, study_name="xgboost")

print("Optimización completada.")

### 7.2 Construcción de modelos con los mejores hiperparámetros


In [ ]:
# Instanciar cada modelo con los mejores hiperparámetros encontrados por Optuna
print("Construyendo LightGBM...")
lgbm_model = build_lgbm(lgbm_study.best_params)

print("Construyendo CatBoost...")
cat_model  = build_catboost(cat_study.best_params)

print("Construyendo XGBoost...")
xgb_model  = build_xgb(xgb_study.best_params)

### 7.3 Generación de predicciones OOF y sobre test
Entrenamos cada modelo con validación cruzada y guardamos:
- Las predicciones OOF (para el stacking).
- Las predicciones sobre test (para la submission final).


In [ ]:
# Generar predicciones OOF y sobre test para cada modelo base
# Las predicciones OOF se usarán como features del metamodelo (stacking)
print("Generando predicciones OOF y sobre test...")

lgbm_res = generate_oof_and_test_preds(
    lgbm_model, train[MODEL_FEATURES], train[TARGET], test[MODEL_FEATURES])

cat_res  = generate_oof_and_test_preds(
    cat_model,  train[MODEL_FEATURES], train[TARGET], test[MODEL_FEATURES])

xgb_res  = generate_oof_and_test_preds(
    xgb_model,  train[MODEL_FEATURES], train[TARGET], test[MODEL_FEATURES])

## 9. Evaluación y generación de predicciones

Evaluamos el rendimiento de cada modelo base y del metamodelo final.
Las métricas principales son:
- **AUC-ROC**: capacidad discriminativa global del modelo.
- **Average Precision (AP)**: útil cuando hay desbalanceo de clases.
- **Matriz de confusión**: distribución de verdaderos/falsos positivos y negativos.


### 8.1 Importancia de características
Visualizamos las variables más relevantes para cada modelo base.
Esto ayuda a interpretar qué factores influyen más en la predicción del churn.


In [ ]:
# Visualizar las 20 variables más importantes para cada modelo base
# Se promedian las importancias de los modelos entrenados en cada fold
plot_feature_importance(lgbm_res["models"], MODEL_FEATURES, top_n=20,
                        title="LightGBM Feature Importance")
plot_feature_importance(cat_res["models"],  MODEL_FEATURES, top_n=20,
                        title="CatBoost Feature Importance")
plot_feature_importance(xgb_res["models"],  MODEL_FEATURES, top_n=20,
                        title="XGBoost Feature Importance")

### 8.2 Evaluación de modelos base (OOF)
Evaluamos cada modelo sobre sus predicciones OOF, que son una estimación
honesta del rendimiento en datos no vistos durante el entrenamiento.


In [ ]:
# Evaluar cada modelo base sobre sus predicciones OOF
# Las predicciones OOF son honestas: cada muestra fue predicha por un modelo que no la vio
evaluate_oof_predictions(train[TARGET], lgbm_res["oof_preds"], model_name="LightGBM")
evaluate_oof_predictions(train[TARGET], cat_res["oof_preds"],  model_name="Catboost")
evaluate_oof_predictions(train[TARGET], xgb_res["oof_preds"],  model_name="Xgboost")

### 8.3 Stacking: construcción del conjunto de features del metamodelo
Combinamos las predicciones OOF de los tres modelos base y añadimos
features derivadas (media, máximo, diferencias entre modelos).


In [ ]:
# Construir el DataFrame de nivel 1 para el metamodelo
# stack_train: predicciones OOF de los modelos base (sin data leakage)
stack_train = pd.DataFrame({
    "lgbm_pred": lgbm_res["oof_preds"],
    "cat_pred":  cat_res["oof_preds"],
    "xgb_pred":  xgb_res["oof_preds"]
})

# stack_test: predicciones sobre test de los modelos base
stack_test = pd.DataFrame({
    "lgbm_pred": lgbm_res["test_preds"],
    "cat_pred":  cat_res["test_preds"],
    "xgb_pred":  xgb_res["test_preds"]
})

In [ ]:
# Enriquecer los DataFrames de stacking con features derivadas
# (media, máximo, mínimo, desviación y diferencias entre modelos)
print("Añadiendo features de stacking...")
stack_train = add_stack_features(stack_train)
stack_test  = add_stack_features(stack_test)

### 8.4 Entrenamiento del metamodelo
Entrenamos la regresión logística sobre las predicciones OOF del nivel 1.


In [ ]:
# Entrenar el metamodelo (regresión logística) sobre las predicciones del nivel 1
print("Entrenando metamodelo...")
meta_res = train_meta_model(stack_train, train[TARGET], stack_test)

### 8.5 Generación de la submission
Generamos el fichero de predicciones en el formato correspondiente.


In [ ]:
# Crear el DataFrame de submission con las predicciones del metamodelo
test_ids = test["id"].copy()
submission = pd.DataFrame({
    "id":         test_ids,
    "prediction": meta_res["test_preds"]  # Probabilidades de churn del metamodelo
})

# Guardar el fichero de submission en formato CSV
submission.to_csv("./data/submission.csv", index=False)

In [ ]:
# Previsualizar las primeras filas del fichero de submission
submission.head(15)

## Conclusiones y Lecciones Aprendidas

¡Felicidades! Has completado un proyecto completo de ML. Aquí van algunas reflexiones educativas:

### Lo que aprendiste:
- **EDA salva vidas**: Entender datos previene errores costosos.
- **Modelos no son black boxes**: Analiza importancia de features para explicar predicciones.
- **Itera y mejora**: Un primer modelo rara vez es el mejor; optimiza y evalúa.
- **Aplicación real**: Este mismo flujo se usa en empresas para retener clientes, predecir fraudes, etc.

### Próximos pasos:
- **Despliega el modelo**: Usa Flask o FastAPI para una API web.
- **Experimenta más**: Prueba otros datasets (e.g., de Kaggle) o algoritmos (e.g., redes neuronales).
- **Ética en ML**: Considera sesgos (e.g., ¿el modelo discrimina por género?). Siempre audita.
